In [34]:
# Testing the geoapify replacement for google maps api since osm is washed
import pandas as pd
import requests
import time
import json
from dotenv import load_dotenv, dotenv_values
from pathlib import Path
import os
from fuzzywuzzy import fuzz

In [17]:
env_path = Path(os.path.abspath('')).resolve().parent / "secrets.env"
env_vars = dotenv_values(env_path)
geoapify_key = env_vars['GEOAPIFY_KEY']

In [21]:
df = pd.read_csv("data_2026-03-02_2026-03-08.csv")
# Filter out bad coordinates
df = df.dropna(subset = ['longitude', 'latitude'])
df = df[(df['longitude'] != 0) & (df['latitude'] != 0)]

# Clean and combine address data for later processing
df['street_name'] = df['street_name'].apply(lambda x : x.title())
df['address'] = df.apply(lambda x : str(x['house_number']) + ' ' + x['street_name'] ,axis = 1)

df = df.iloc[1:100]

df = df[
    ['inspection_type',
     'job_id',
     'job_progress',
     'house_number',
     'street_name',
     'address',
     'zip_code',
     'latitude',
     'longitude',
     'result',
     'approved_date',
     'nta'
     ]
]

df.columns = df.columns.str.lower() + '_interdata'
df.head()

,inspection_type_interdata,job_id_interdata,job_progress_interdata,house_number_interdata,street_name_interdata,address_interdata,zip_code_interdata,latitude_interdata,longitude_interdata,result_interdata,approved_date_interdata,nta_interdata
2,Initial,PC8673353,1,88-47,187 Street,88-47 187 Street,11423.0,40.713865,-73.774740,Rat Activity,2026-03-03T14:36:31.000,Hollis
3,Initial,PC8671158,1,675,Riverside Drive,675 Riverside Drive,10031.0,40.826757,-73.952242,Failed for Rat Act,2026-03-04T09:46:11.000,Hamilton Heights-Sugar Hill
4,Initial,PC8672570,1,1603,President Street,1603 President Street,11213.0,40.667220,-73.935026,Failed for Other R,2026-03-03T08:42:22.000,Crown Heights (South)
5,Initial,PC8662766,1,768,Putnam Avenue,768 Putnam Avenue,11221.0,40.686198,-73.931088,Rat Activity,2026-03-03T10:14:31.000,Bedford-Stuyvesant (East)
6,Treatments,PC8589110,4,73,Steuben Street,73 Steuben Street,11205.0,40.695219,-73.963422,Bait applied,2026-03-02T13:38:21.000,Clinton Hill


In [22]:

categories = "catering.restaurant,commercial.food_and_drink,catering.fast_food,catering.food_court,catering.bar"
radius = "50"
max_locations_returned = "5"

all_restaurants_df = pd.DataFrame()

for _ , row in df.iterrows():

    time.sleep(2)
    longitude = str(row['longitude_interdata'])
    latitude = str(row['latitude_interdata'])

    url = f"https://api.geoapify.com/v2/places?categories={categories}&filter=circle:{longitude},{latitude},{radius}&bias=proximity:{longitude},{latitude}&limit={max_locations_returned}&apiKey={geoapify_key}"

    payload = {}
    headers = {}

    response = requests.request("GET", url, headers=headers, data=payload)

    res = json.loads(response.text)
    print(res)
    try:
        local_restaurant_df = pd.DataFrame([
            {
                **feature["properties"],
                "geo_lon": feature["geometry"]["coordinates"][0],
                "geo_lat": feature["geometry"]["coordinates"][1],
            }
            for feature in res["features"]
        ])
        
        # selected_columns = ['name', 'county', 'city', 'postcode', 'district', 'suburb', 'quarter', 'street', 'housenumber','lon','lat', 'formatted', 'catering']
        # local_restaurant_df = local_restaurant_df[selected_columns]

        local_restaurant_df['inspection_job_id'] = row['inspection_type_interdata']
        local_restaurant_df['job_id_interdata'] = row['job_id_interdata']
        local_restaurant_df['job_progress_interdata'] = row['job_progress_interdata']
        local_restaurant_df['house_number_interdata'] = row['house_number_interdata']
        local_restaurant_df['street_name_interdata'] = row['street_name_interdata']
        local_restaurant_df['address_interdata'] = row['address_interdata']
        local_restaurant_df['zip_code_interdata'] = row['zip_code_interdata']
        local_restaurant_df['latitude_interdata'] = row['latitude_interdata']
        local_restaurant_df['longitude_interdata'] = row['longitude_interdata']
        local_restaurant_df['result_interdata'] = row['result_interdata']
        local_restaurant_df['approved_date_interdata'] = row['approved_date_interdata']
        local_restaurant_df['nta_interdata'] = row['nta_interdata']
        
        all_restaurants_df = pd.concat([all_restaurants_df, local_restaurant_df])
    except Exception as e:
        print(e)
        # Move on if no data is found
        continue
    

{'type': 'FeatureCollection', 'features': []}
{'type': 'FeatureCollection', 'features': []}
{'type': 'FeatureCollection', 'features': []}
{'type': 'FeatureCollection', 'features': []}
{'type': 'FeatureCollection', 'features': []}
{'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'properties': {'name': 'Hello, Yam!', 'country': 'United States', 'country_code': 'us', 'state': 'New York', 'county': 'New York County', 'city': 'New York', 'postcode': '10009', 'district': 'East Village', 'suburb': 'Manhattan', 'street': 'East 9th Street', 'housenumber': '4432', 'iso3166_2': 'US-NY', 'lon': -73.9829136, 'lat': 40.727499, 'state_code': 'NY', 'formatted': 'Hello, Yam!, 4432 East 9th Street, New York, NY 10009, United States of America', 'address_line1': 'Hello, Yam!', 'address_line2': '4432 East 9th Street, New York, NY 10009, United States of America', 'categories': ['catering', 'catering.fast_food', 'vegan'], 'details': ['details', 'details.catering', 'details.contact'], 'datasou

In [23]:
print(all_restaurants_df.columns)
all_restaurants_df.head()

Index(['inspection_job_id', 'job_id_interdata', 'job_progress_interdata',
       'house_number_interdata', 'street_name_interdata', 'address_interdata',
       'zip_code_interdata', 'latitude_interdata', 'longitude_interdata',
       'result_interdata', 'approved_date_interdata', 'nta_interdata', 'name',
       'country', 'country_code', 'state', 'county', 'city', 'postcode',
       'district', 'suburb', 'street', 'housenumber', 'iso3166_2', 'lon',
       'lat', 'state_code', 'formatted', 'address_line1', 'address_line2',
       'categories', 'details', 'datasource', 'website', 'contact', 'catering',
       'distance', 'place_id', 'geo_lon', 'geo_lat', 'opening_hours',
       'description', 'commercial', 'facilities', 'brand', 'brand_details',
       'quarter', 'building', 'name_other'],
      dtype='object')


,inspection_job_id,job_id_interdata,job_progress_interdata,house_number_interdata,street_name_interdata,address_interdata,zip_code_interdata,latitude_interdata,longitude_interdata,result_interdata,...,geo_lat,opening_hours,description,commercial,facilities,brand,brand_details,quarter,building,name_other
0,Compliance,PC8654073,2,447,East 9 Street,447 East 9 Street,10009.0,40.727378,-73.982934,Rat Activity,...,40.727499,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Compliance,PC8654073,2,447,East 9 Street,447 East 9 Street,10009.0,40.727378,-73.982934,Rat Activity,...,40.727320,"Mo-Su 12:00-15:30, Mo-Sa 16:15-20:00",Vegan sweet shop.,{'type': 'bakery'},NaN,NaN,NaN,NaN,NaN,NaN
2,Compliance,PC8654073,2,447,East 9 Street,447 East 9 Street,10009.0,40.727378,-73.982934,Rat Activity,...,40.727228,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Compliance,PC8654073,2,447,East 9 Street,447 East 9 Street,10009.0,40.727378,-73.982934,Rat Activity,...,40.727541,Mo-Su 11:00-22:00,NaN,NaN,{'takeaway': True},NaN,NaN,NaN,NaN,NaN
4,Compliance,PC8654073,2,447,East 9 Street,447 East 9 Street,10009.0,40.727378,-73.982934,Rat Activity,...,40.727363,Mo-Su 11:00-22:30,NaN,NaN,"{'takeaway': True, 'delivery': True}",NaN,NaN,NaN,NaN,NaN


In [44]:
selected_columns = ['name', 'county', 'city', 'postcode', 'district', 'suburb', 'housenumber', 'street', 'lon','lat', 'formatted', 'catering', 'commercial','house_number_interdata', 'street_name_interdata', 'address_interdata', 'result_interdata']
geo_rest_df = all_restaurants_df[selected_columns]
geo_rest_df.head()

,name,county,city,postcode,district,suburb,housenumber,street,lon,lat,formatted,catering,commercial,house_number_interdata,street_name_interdata,address_interdata,result_interdata
0,"Hello, Yam!",New York County,New York,10009,East Village,Manhattan,4432,East 9th Street,-73.982914,40.727499,"Hello, Yam!, 4432 East 9th Street, New York, N...","{'cuisine': 'japanese;sweet_potato', 'diet': {...",NaN,447,East 9 Street,447 East 9 Street,Rat Activity
1,Confectionery!,New York County,New York,10003,East Village,Manhattan,440,East 9th Street,-73.983075,40.727320,"Confectionery!, 440 East 9th Street, New York,...",NaN,{'type': 'bakery'},447,East 9 Street,447 East 9 Street,Rat Activity
2,Doc Hollidays,New York County,New York,10009,Alphabet City,Manhattan,141,Avenue A,-73.982885,40.727228,"Doc Hollidays, 141 Avenue A, New York, NY 1000...",NaN,NaN,447,East 9 Street,447 East 9 Street,Rat Activity
3,Poke N' Roll,New York County,New York,10009,East Village,Manhattan,441,East 9th Street,-73.983043,40.727541,"Poke N' Roll, 441 East 9th Street, New York, N...",{'cuisine': 'hawaiian'},NaN,447,East 9 Street,447 East 9 Street,Rat Activity
4,Tacos Morelos,New York County,New York,10009,East Village,Manhattan,438,East 9th Street,-73.983187,40.727363,"Tacos Morelos, 438 East 9th Street, New York, ...",NaN,NaN,447,East 9 Street,447 East 9 Street,Rat Activity


In [54]:
# requested_places_saved = geo_rest_df.copy()

# requested_places_saved['ratio'] = requested_places_saved.apply(lambda x : fuzz.WRatio(x['formatted'], x['address_interdata']), axis = 1)
# requested_places_saved = requested_places_saved[requested_places_saved['ratio'] >= 80]

# # Check number in address for final comparison
# requested_places_saved['split_check'] = requested_places_saved.apply(lambda x : x['formatted'].split()[1] == x['address_interdata'].split()[0], axis = 1)
# # requested_places_saved = requested_places_saved[requested_places_saved['split_check'] == True]

# # # requested_places_saved = requested_places_saved.drop(columns = ["ratio", "split_check", "displayName", "formatted", "location"])

# # requested_places_saved[['lon', 'lat']] = requested_places_saved[['lon', 'lat']].round(6)

# requested_places_saved.to_csv("requested_places_saved.csv", index = False)
# requested_places_saved

import re
from rapidfuzz import fuzz

def normalise_address(address: str) -> str:
    if not isinstance(address, str):
        return ""
    address = address.lower().strip()
    address = re.sub(r'(\d+)(st|nd|rd|th)\b', r'\1', address)  # 5th → 5
    address = re.sub(r'\s+', ' ', address)                      # collapse whitespace
    return address

def addresses_match(formatted: str, interdata: str, threshold: int = 88) -> bool:
    parts = formatted.split(",")
    addr_segment = parts[1].strip() if len(parts) > 1 else formatted
    norm_formatted = normalise_address(addr_segment)
    norm_interdata = normalise_address(interdata)
    score = fuzz.token_sort_ratio(norm_formatted, norm_interdata)
    return score >= threshold

requested_places_saved = geo_rest_df.copy()

requested_places_saved['ratio'] = requested_places_saved.apply(lambda x: fuzz.WRatio(x['formatted'], x['address_interdata']), axis=1)
requested_places_saved = requested_places_saved[requested_places_saved['ratio'] >= 80]

requested_places_saved['split_check'] = requested_places_saved.apply(lambda x: addresses_match(x['formatted'], x['address_interdata']), axis=1)

# requested_places_saved.to_csv("requested_places_saved.csv", index=False)
requested_places_saved

,name,county,city,postcode,district,suburb,housenumber,street,lon,lat,formatted,catering,commercial,house_number_interdata,street_name_interdata,address_interdata,result_interdata,ratio,split_check
0,"Hello, Yam!",New York County,New York,10009,East Village,Manhattan,4432,East 9th Street,-73.982914,40.727499,"Hello, Yam!, 4432 East 9th Street, New York, N...","{'cuisine': 'japanese;sweet_potato', 'diet': {...",NaN,447,East 9 Street,447 East 9 Street,Rat Activity,85.500000,False
1,Confectionery!,New York County,New York,10003,East Village,Manhattan,440,East 9th Street,-73.983075,40.727320,"Confectionery!, 440 East 9th Street, New York,...",NaN,{'type': 'bakery'},447,East 9 Street,447 East 9 Street,Rat Activity,85.500000,True
3,Poke N' Roll,New York County,New York,10009,East Village,Manhattan,441,East 9th Street,-73.983043,40.727541,"Poke N' Roll, 441 East 9th Street, New York, N...",{'cuisine': 'hawaiian'},NaN,447,East 9 Street,447 East 9 Street,Rat Activity,85.500000,True
4,Tacos Morelos,New York County,New York,10009,East Village,Manhattan,438,East 9th Street,-73.983187,40.727363,"Tacos Morelos, 438 East 9th Street, New York, ...",NaN,NaN,447,East 9 Street,447 East 9 Street,Rat Activity,85.500000,True
0,Tipsy,Kings County,New York,11205,NaN,Brooklyn,584,Myrtle Avenue,-73.961238,40.693875,"Tipsy, 584 Myrtle Avenue, New York, NY 11205, ...",NaN,{'type': 'alcohol'},584,Myrtle Avenue,584 Myrtle Avenue,Bait applied,90.000000,True
1,Moot Bar,Kings County,New York,11205,NaN,Brooklyn,579,Myrtle Avenue,-73.960971,40.694132,"Moot Bar, 579 Myrtle Avenue, New York, NY 1120...",NaN,NaN,584,Myrtle Avenue,584 Myrtle Avenue,Bait applied,85.500000,True
2,Nathan's,Kings County,New York,11205,NaN,Brooklyn,569,Myrtle Avenue,-73.961553,40.694154,"Nathan's, 569 Myrtle Avenue, New York, NY 1120...",{'cuisine': 'hot_dog'},NaN,584,Myrtle Avenue,584 Myrtle Avenue,Bait applied,85.500000,True
3,The Emerson,Kings County,New York,11205,NaN,Brooklyn,561,Myrtle Avenue,-73.961802,40.694054,"The Emerson, 561 Myrtle Avenue, New York, NY 1...",NaN,NaN,584,Myrtle Avenue,584 Myrtle Avenue,Bait applied,85.500000,True
4,Kingston Bakery Brooklyn,Kings County,New York,11205,NaN,Brooklyn,585,Myrtle Avenue,-73.960735,40.694161,"Kingston Bakery Brooklyn, 585 Myrtle Avenue, N...",{'cuisine': 'jamaican'},NaN,584,Myrtle Avenue,584 Myrtle Avenue,Bait applied,85.500000,True
0,Di Fara,Kings County,New York,11249,NaN,Brooklyn,103,North 3rd Street,-73.961798,40.717079,"Di Fara, 103 North 3rd Street, New York, NY 11...",{'cuisine': 'pizza'},NaN,91,North 4 Street,91 North 4 Street,Bait applied,85.500000,False


In [31]:
geo_rest_df

,name,county,city,postcode,district,suburb,housenumber,street,lon,lat,formatted,catering,commercial,house_number_interdata,street_name_interdata
0,"Hello, Yam!",New York County,New York,10009,East Village,Manhattan,4432,East 9th Street,-73.982914,40.727499,"Hello, Yam!, 4432 East 9th Street, New York, N...","{'cuisine': 'japanese;sweet_potato', 'diet': {...",NaN,447,East 9 Street
1,Confectionery!,New York County,New York,10003,East Village,Manhattan,440,East 9th Street,-73.983075,40.727320,"Confectionery!, 440 East 9th Street, New York,...",NaN,{'type': 'bakery'},447,East 9 Street
2,Doc Hollidays,New York County,New York,10009,Alphabet City,Manhattan,141,Avenue A,-73.982885,40.727228,"Doc Hollidays, 141 Avenue A, New York, NY 1000...",NaN,NaN,447,East 9 Street
3,Poke N' Roll,New York County,New York,10009,East Village,Manhattan,441,East 9th Street,-73.983043,40.727541,"Poke N' Roll, 441 East 9th Street, New York, N...",{'cuisine': 'hawaiian'},NaN,447,East 9 Street
4,Tacos Morelos,New York County,New York,10009,East Village,Manhattan,438,East 9th Street,-73.983187,40.727363,"Tacos Morelos, 438 East 9th Street, New York, ...",NaN,NaN,447,East 9 Street
0,Diyaanaat NY,Queens County,New York,11432,NaN,Queens,160-02,160th Street,-73.801612,40.707687,"Diyaanaat NY, 160-02 160th Street, New York, N...",NaN,"{'type': 'butcher', 'level': 0}",87-82,160 Street
1,Sybil's Restaurant & Bakery,Queens County,New York,11432,NaN,Queens,159-24,Hillside Avenue,-73.801885,40.707809,"Sybil's Restaurant & Bakery, 159-24 Hillside A...",NaN,NaN,87-82,160 Street
0,Tipsy,Kings County,New York,11205,NaN,Brooklyn,584,Myrtle Avenue,-73.961238,40.693875,"Tipsy, 584 Myrtle Avenue, New York, NY 11205, ...",NaN,{'type': 'alcohol'},584,Myrtle Avenue
1,Moot Bar,Kings County,New York,11205,NaN,Brooklyn,579,Myrtle Avenue,-73.960971,40.694132,"Moot Bar, 579 Myrtle Avenue, New York, NY 1120...",NaN,NaN,584,Myrtle Avenue
2,Nathan's,Kings County,New York,11205,NaN,Brooklyn,569,Myrtle Avenue,-73.961553,40.694154,"Nathan's, 569 Myrtle Avenue, New York, NY 1120...",{'cuisine': 'hot_dog'},NaN,584,Myrtle Avenue


In [32]:
all_restaurants_df.to_csv("all_restaurants_geoapify.csv", index = False)